In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer, DataCollatorForMultipleChoice
from datasets import Dataset

try: 
    train = pd.read_csv('../data/train.csv') # local
except: 
    train = pd.read_csv('train.csv') # colab

/home/phoenix/Projects/college stuff/smart-mcq-solver/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# q1
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
print(label_map[train.iloc[150]['answer']])

2


In [3]:
# q2 - character length of formatted string for row 0 option b
row_0 = train.iloc[0]
formatted = str(row_0['prompt']) + " [SEP] " + str(row_0['B'])
print(len(formatted))

407


In [4]:
from transformers import AutoTokenizer
# q3

row_0 = train.iloc[0]
prompt = str(row_0['prompt'])
choices = [str(row_0[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
inputs = [prompt + " [SEP] " + c for c in choices]

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenized = tokenizer(inputs, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
input_ids = tokenized['input_ids'].unsqueeze(0)
print(input_ids.shape[1])

5


In [ ]:
# 4
first_16 = train.iloc[:16]
all_inputs = []
for idx, row in first_16.iterrows():
    p = str(row['prompt'])
    for opt in ['A', 'B', 'C', 'D', 'E']:
        all_inputs.append(p + " [SEP] " + str(row[opt]))
tokenized_16 = tokenizer(all_inputs, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
input_ids_16 = tokenized_16['input_ids'].view(16, 5, 128)
print(input_ids_16.numel())

10240


In [ ]:
# q5 
model_mc = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
row_0 = train.iloc[0]
prompt = str(row_0['prompt'])
choices = [str(row_0[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
inputs = [prompt + " [SEP] " + c for c in choices]
tokenized = tokenizer(inputs, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
input_ids = tokenized['input_ids'].unsqueeze(0)
attention_mask = tokenized['attention_mask'].unsqueeze(0)
outputs = model_mc(input_ids=input_ids, attention_mask=attention_mask)
print(outputs.logits.shape[-1])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 230.69it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consi

5


In [ ]:
# q6 
labels = torch.tensor([label_map[row_0['answer']]])
outputs = model_mc(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
print(outputs.loss.dim())

0


In [8]:
# !pip install peft transformers accelerate datasets torch

In [ ]:
# q7
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
model_peft = get_peft_model(model, lora_config)
sum(p.numel() for p in model_peft.parameters() if p.requires_grad)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 478.24it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consi

295681


In [ ]:
# q8
first_100 = train.iloc[:100]
dataset_dict = {'input_ids': [], 'attention_mask': [], 'labels': []}
for idx, row in first_100.iterrows():
    p = str(row['prompt'])
    choices_texts = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
    inputs = [p + " [SEP] " + c for c in choices_texts]
    tokenized = tokenizer(inputs, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    dataset_dict['input_ids'].append(tokenized['input_ids'])
    dataset_dict['attention_mask'].append(tokenized['attention_mask'])
    dataset_dict['labels'].append(label_map[row['answer']])

dataset_hf = Dataset.from_dict(dataset_dict)
len(dataset_hf[0]['input_ids'])

5


In [ ]:
# q9
first_32 = train.iloc[:32]
dataset_dict_32 = {'input_ids': [], 'attention_mask': [], 'labels': []}
for idx, row in first_32.iterrows():
    p = str(row['prompt'])
    choices_texts = [str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
    inputs = [p + " [SEP] " + c for c in choices_texts]
    tokenized = tokenizer(inputs, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    dataset_dict_32['input_ids'].append(tokenized['input_ids'])
    dataset_dict_32['attention_mask'].append(tokenized['attention_mask'])
    dataset_dict_32['labels'].append(label_map[row['answer']])

train_dataset = Dataset.from_dict(dataset_dict_32)

model_ft = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
lora_config_ft = LoraConfig(
    r=8, lora_alpha=16, target_modules=["query", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
model_peft_ft = get_peft_model(model_ft, lora_config_ft)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
)

trainer = Trainer(
    model=model_peft_ft,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

trainer.train()
trainer.state.global_step

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 184.87it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consi

4


In [ ]:
# q10
row_0 = train.iloc[0]
p = str(row_0['prompt'])
choices = [str(row_0[opt]) for opt in ['A', 'B', 'C', 'D', 'E']]
inputs = [p + " [SEP] " + c for c in choices]
tokenized = tokenizer(inputs, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
input_ids = tokenized['input_ids'].unsqueeze(0)
attention_mask = tokenized['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs = model_peft_ft(input_ids=input_ids, attention_mask=attention_mask)
probs = F.softmax(outputs.logits, dim=-1)
print(round(probs[0][4].item(), 4))

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


0.2034
